# 지금 롱/숏/중립? — BTC 선물 실시간 의사결정

단기 타임프레임(수 시간~수일) 기준. 11개 신호를 가중 융합하여 LONG / SHORT / NEUTRAL + 신뢰도를 산출.

모든 점수는 `-1.0`(강한 숏) ~ `+1.0`(강한 롱), 신뢰도는 `0~1`. **확률적 보조 신호**이며 시장 방향을 확정 예측하지 않습니다.

In [ ]:
import pandas as pd
from datetime import datetime, timedelta, timezone

from crypto_analysis.collectors import deribit, exchanges, onchain
from crypto_analysis.collectors.macro import macro_panel
from crypto_analysis.config import PERPETUAL
from crypto_analysis.decision import decide, format_report
from crypto_analysis.engine import fuse, DEFAULT_WEIGHTS
from crypto_analysis.signals.basis import compute as basis_compute
from crypto_analysis.signals.funding import compute as funding_compute
from crypto_analysis.signals.gex import compute as gex_compute
from crypto_analysis.signals.iv_skew import compute as iv_skew_compute
from crypto_analysis.signals.macro import compute as macro_compute
from crypto_analysis.signals.news import compute as news_compute
from crypto_analysis.signals.oi import compute as oi_compute
from crypto_analysis.signals.onchain import compute as onchain_compute
from crypto_analysis.signals.option_skew import compute as option_skew_compute
from crypto_analysis.signals.orderbook import compute as orderbook_compute
from crypto_analysis.signals.spot_futures import compute as spot_futures_compute

now = datetime.now(tz=timezone.utc)
start_ms = int((now - timedelta(days=7)).timestamp() * 1000)
end_ms = int(now.timestamp() * 1000)
now

## 1. 원시 입력 수집 (Deribit / Binance / mempool.space / yfinance)

In [ ]:
book = deribit.book_summary_by_currency('BTC', 'future')
perp_ticker = deribit.ticker(PERPETUAL)
perp_mark = float(perp_ticker.get('mark_price') or 0)
perp_ob = deribit.order_book(PERPETUAL, depth=20)
options = deribit.option_book_summary('BTC')
rv = deribit.historical_volatility('BTC')
funding = deribit.funding_rate_history(PERPETUAL, start_ms, end_ms)
if not funding.empty:
    funding['funding_rate'] = funding.get('interest_1h', funding.get('funding_rate'))
binance_spot = exchanges.binance_klines('BTCUSDT', '1m', limit=5)
spot_price = float(binance_spot['close'].iloc[-1]) if not binance_spot.empty else 0.0
mempool = onchain.mempool_snapshot()
fees = onchain.fees_recommended()
hr = onchain.hashrate_3d()
mp = macro_panel(days=45)
print(f'perp={perp_mark}, spot={spot_price}, options={len(options)}, macro_rows={len(mp)}')

## 2. Macro / 뉴스 브리프 (선택)

WebSearch로 최근 24시간 주요 헤드라인을 구조화한 뒤 아래 dict에 넣으세요. 뉴스는 노이즈가 커서 기본 가중치 4%로 제한됩니다.

In [ ]:
news_brief = {
    'window_hours': 24,
    'events': [
        # {'headline': '...', 'bias': 'long|short|neutral', 'weight': 0.0-1.0, 'rationale': '...'},
    ],
    'macro_calendar': [],
}

## 3. ATM IV 조회 (iv_skew 신호용)

In [ ]:
atm_iv_val = None
if not options.empty and spot_price:
    soon = options[options['expiry'] > now].sort_values('expiry').head(60)
    if not soon.empty:
        soon = soon.assign(abs_delta=(soon['strike'] - spot_price).abs())
        near_atm = soon.sort_values(['expiry', 'abs_delta']).iloc[0]
        atm_iv_val = float(near_atm.get('mark_iv') or 0) or None
atm_iv_val

## 4. 신호 11종 계산 & 융합

In [ ]:
ticker_hist = pd.DataFrame([{
    'ts': now,
    'mark_price': perp_mark,
    'open_interest': perp_ticker.get('open_interest', 0),
}])

signals = [
    funding_compute(funding),
    basis_compute(book, spot_price),
    oi_compute(ticker_hist),
    iv_skew_compute(rv, atm_iv_val),
    spot_futures_compute(perp_mark, spot_price),
    orderbook_compute(perp_ob, levels=10),
    option_skew_compute(options, spot_price),
    gex_compute(options, spot_price, dte_max=60),
    onchain_compute(mempool, fees, hr),
    macro_compute(mp),
    news_compute(news_brief),
]
for s in signals:
    print(f'{s.name:<13} score={s.score:+.3f} conf={s.confidence:.2f} :: {s.rationale[:110]}')

In [ ]:
result = fuse(signals)
verdict = decide(result)
print(format_report(result, verdict))

## 5. 사용자 가중치 오버라이드 (선택)

In [ ]:
# 예: 옵션 신호를 강조, 뉴스는 끔
custom = dict(DEFAULT_WEIGHTS)
custom['option_skew'] = 0.20
custom['gex'] = 0.12
custom['news'] = 0.0
# 재정규화는 엔진이 자동으로 수행.

result2 = fuse(signals, weights=custom)
verdict2 = decide(result2)
print(format_report(result2, verdict2))